# Tích hợp Neural Demapper (Autoencoder) vào BICM-ID (Extended Hamming 8,4)---Notebook này thử nghiệm việc xây dựng một hệ thống **Deep Learning-based BICM-ID**. Sử dụng mã **Extended Hamming (8,4)** và bộ giải mã **SISO Dual Decoder**.- **Neural Modulator**: Mạng học cách chuyển bit thành điểm IQ (Es=10).- **Neural Demapper**: Mạng giải điều chế lặp nhận **Tín hiệu IQ**, **A-priori LLR**, và **N0** để xuất ra Extrinsic LLR.

In [ ]:
import sys, os, timeimport numpy as npimport tensorflow as tffrom tensorflow.keras import layers, Modelimport matplotlib.pyplot as pltplt.rcParams.update({'font.size': 12, 'figure.figsize': (12, 7)})%matplotlib inline# Add project rootsys.path.insert(0, os.path.abspath('..'))import warningswarnings.filterwarnings('ignore')# ── Hằng số hệ thống (PHẢI KHỚP VỚI sim_ham_bicmid.py) ──BPS = 4          # bits per symbol (16-QAM)CODE_RATE = 0.5  # Extended Hamming (8,4)# QUAN TRỌNG: build_constellation(Es=10) chia cho sqrt(10) → Es thực = 1.0# Neural Modulator PHẢI output Es = 1.0 để khớp!ES = 1.0def snr_to_N0(dB):    """Tính N0 từ dB, CHÍNH XÁC như sim_ham_bicmid.py"""    SNR_lin = 10**(dB / 10.0) * BPS    N0_uncoded = 1.0 / SNR_lin    return N0_uncoded / CODE_RATE

## 1. Xây dựng Kiến trúc Mạng Nơ-ron

In [ ]:
# 1. Neural Modulator (Es = 1.0, khởi tạo bằng MSEW Mapping)class NeuralModulator(layers.Layer):    def __init__(self, m_bits=4, trainable_constellation=False, **kwargs):        super().__init__(**kwargs)        self.m_bits = m_bits                # Lấy MSEW mapping từ hệ thống gốc        from python_code.modulation import build_constellation        maprule_msew = [11, 2, 1, 12, 4, 9, 10, 3, 5, 16, 15, 6, 14, 7, 8, 13]        S = build_constellation(16, maprule_msew, ES)        init_points = np.stack([S.real, S.imag], axis=-1).astype(np.float32)                self.constellation_points = tf.Variable(            initial_value=init_points,            trainable=trainable_constellation,  # False = đóng băng (để giữ Iterative Gain)            name="constellation_points"        )                import itertools        self.all_bits = tf.constant(            list(itertools.product([0, 1], repeat=m_bits)), dtype=tf.float32)    def call(self, bits, training=False):        # bits: (Batch, 4) chứa float 0.0 hoặc 1.0        # Chuyển nhị phân thành index thập phân (MSB first)        indices = tf.cast(            bits[:, 0] * 8 + bits[:, 1] * 4 + bits[:, 2] * 2 + bits[:, 3],             tf.int32        )                # Lấy điểm IQ        iq = tf.gather(self.constellation_points, indices)                # Chuẩn hóa để luôn đảm bảo Es = 1.0 (phòng khi bạn bật trainable=True sau này)        avg_power = tf.reduce_mean(tf.reduce_sum(tf.square(self.constellation_points), axis=1))        return iq * tf.sqrt(ES / (avg_power + 1e-8))# 2. Neural Iterative Demapper (Sub-network, Extrinsic Principle)class NeuralDemapper(layers.Layer):    def __init__(self, m_bits=4, **kwargs):        super().__init__(**kwargs)        self.m_bits = m_bits        self.sub_nets = [self._build_net() for _ in range(m_bits)]    def _build_net(self):        return tf.keras.Sequential([            layers.Dense(128, activation='relu'),            layers.Dense(64, activation='relu'),            layers.Dense(64, activation='relu'),            layers.Dense(1, activation='linear')        ])    def call(self, inputs):        rx_iq, llr_apriori, N0_input = inputs                # Chuẩn hóa N0 cho NN (log scale, mean ~0)        log_N0 = tf.math.log(N0_input + 1e-8) / 3.0                ext_llrs = []        for i in range(self.m_bits):            mask = tf.one_hot(i, self.m_bits, on_value=0.0, off_value=1.0)            la_masked = llr_apriori * mask            la_clamped = tf.clip_by_value(la_masked, -10.0, 10.0)                        # Input: [IQ(2) + masked_LLR(4) + log_N0(1)] = 7            concat_in = tf.concat([rx_iq, la_clamped, log_N0], axis=-1)            le_i = self.sub_nets[i](concat_in)            ext_llrs.append(le_i)                    return tf.concat(ext_llrs, axis=-1)

## 2. Xây dựng mô hình End-to-End cho quá trình TrainSử dụng **Simulated A-priori** và công thức N0 **CHÍNH XÁC** từ hệ thống Convention.

In [ ]:
class AE_BICMID_Trainer(Model):    def __init__(self, m_bits=4):        super().__init__()        self.m_bits = m_bits        self.modulator = NeuralModulator(m_bits)        self.demapper = NeuralDemapper(m_bits)            def call(self, inputs, training=False):        bits, snr_dB, sigma_apriori = inputs                # 1. Phát (Es = 10)        tx_iq = self.modulator(bits, training=training)                # 2. Kênh truyền (Tùy chọn thêm PA)        # Mô hình Rapp PA — chỉ dùng phép toán thực (I/Q), không dùng complex        if getattr(self, 'use_pa', False):            I = tx_iq[:, 0]            Q = tx_iq[:, 1]            r = tf.sqrt(I**2 + Q**2 + 1e-12)                        a_sat = 2.5            p_rapp = 2.0            ratio = r / a_sat            g_r = r / tf.pow(1.0 + tf.pow(ratio, 2.0 * p_rapp), 1.0 / (2.0 * p_rapp))                        # Scale factor: g(r)/r giữ nguyên pha, chỉ nén biên độ            scale = g_r / (r + 1e-12)            tx_iq = tf.stack([I * scale, Q * scale], axis=-1)        # AWGN        N0 = snr_to_N0(snr_dB)  # scalar or (B,1)        sigma_ch = tf.sqrt(N0 / 2.0)  # per real dimension        noise = tf.random.normal(tf.shape(tx_iq)) * tf.reshape(sigma_ch, (-1, 1))        rx_iq = tx_iq + noise                # 3. Giả lập A-priori LLR        bits_bpsk = 2.0 * bits - 1.0        ap_noise = tf.random.normal(tf.shape(bits_bpsk)) * tf.reshape(sigma_apriori, (-1, 1))        ap_rx = bits_bpsk + ap_noise        llr_apriori = 2.0 * ap_rx / (tf.square(tf.reshape(sigma_apriori, (-1, 1))) + 1e-8)                # 20% drop La → buộc mạng học Iteration 1        drop_mask = tf.cast(tf.random.uniform((tf.shape(bits)[0], 1)) > 0.2, tf.float32)        llr_apriori = llr_apriori * drop_mask        # 4. Neural Demapper (nhận N0 thay vì SNR)        N0_input = tf.reshape(N0, (-1, 1))        llr_extrinsic = self.demapper([rx_iq, llr_apriori, N0_input])                return llr_extrinsictrainer = AE_BICMID_Trainer(m_bits=4)trainer.compile(optimizer=tf.keras.optimizers.Adam(1e-3),                loss=tf.keras.losses.BinaryCrossentropy(from_logits=True))print("✅ Mô hình đã sẵn sàng để huấn luyện.")

## 3. Huấn luyện (Training) Mô hình> Với Es=10, N0 khớp Convention, Loss sẽ giảm sâu hơn nhiều.

In [ ]:
def generate_batch(batch_size=2048, m_bits=4):    bits = tf.random.uniform((batch_size, m_bits), minval=0, maxval=2, dtype=tf.int32)    bits = tf.cast(bits, tf.float32)        # SNR range 0-6 dB (khớp trục hoành của conventional BER plot)    snr_dB = tf.random.uniform((batch_size, 1), minval=0.0, maxval=6.0)    sigma_apriori = tf.random.uniform((batch_size, 1), minval=0.3, maxval=5.0)        return bits, snr_dB, sigma_aprioriEPOCHS = 50STEPS_PER_EPOCH = 50LOG_FILE = 'neural_training_log.txt'log_f = open(LOG_FILE, 'w')log_f.write("=== Neural BICM-ID Training Log ===\n")log_f.write(f"Es={ES}, BPS={BPS}, CODE_RATE={CODE_RATE}\n")log_f.write(f"EPOCHS={EPOCHS}, STEPS={STEPS_PER_EPOCH}, BATCH=2048\n\n")losses = []for epoch in range(EPOCHS):    epoch_loss = 0    for _ in range(STEPS_PER_EPOCH):        b_bits, b_snr, b_sigma = generate_batch(2048, 4)        with tf.GradientTape() as tape:            llr_ext = trainer([b_bits, b_snr, b_sigma], training=True)            loss = trainer.compiled_loss(b_bits, llr_ext)                    grads = tape.gradient(loss, trainer.trainable_variables)        trainer.optimizer.apply_gradients(zip(grads, trainer.trainable_variables))        epoch_loss += loss.numpy()            avg_loss = epoch_loss / STEPS_PER_EPOCH    losses.append(avg_loss)    if (epoch+1) % 10 == 0:        msg = f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}"        print(msg)        log_f.write(msg + "\n")        log_f.flush()plt.plot(losses)plt.title('Training Loss (Extrinsic BCE)')plt.grid(True)plt.show()

## 4. Trực quan hóa "Chòm sao Học được" (Learned Constellation)

In [ ]:
import itertoolsall_bits = np.array(list(itertools.product([0, 1], repeat=4)), dtype=np.float32)learned_S = trainer.modulator(all_bits, training=False).numpy()Es_check = np.mean(np.sum(learned_S**2, axis=1))print(f"Learned constellation Es = {Es_check:.4f} (target: {ES})")plt.figure(figsize=(6, 6))plt.scatter(learned_S[:, 0], learned_S[:, 1], c='red', s=100, zorder=5)for i in range(16):    bit_str = "".join(str(int(b)) for b in all_bits[i])    plt.annotate(bit_str, (learned_S[i, 0], learned_S[i, 1]), xytext=(5, 5), textcoords='offset points')plt.title(f'Learned 16-QAM Constellation (Es={Es_check:.2f})')plt.axhline(0, color='black', lw=1); plt.axvline(0, color='black', lw=1)plt.grid(True, alpha=0.3); plt.axis('equal')plt.show()

## 5. Tích hợp trực tiếp vào vòng lặp BICM-ID (Extended Hamming 8,4)

In [ ]:
from python_code.encoders import HammingEncoderfrom python_code.decoders import DualDecoderfrom python_code.interleavers import load_interleaverhamming_m = 3encoder = HammingEncoder(hamming_m)decoder = DualDecoder(encoder.H)alpha_conv = load_interleaver("BIBCM-ID_4096Algeb.mat") num_channel_bits = len(alpha_conv)n_code = encoder.n_codek_info = encoder.k_infoframe_len = num_channel_bits // n_codeinfo_len = frame_len * k_infoscaling_factor = 0.85def simulate_neural_hamming_bicmid_frame(snr_dB, max_iter=10):    u = np.random.randint(0, 2, info_len)    v = encoder.encode_frame(u)    vv = v[alpha_conv]        vv_reshaped = vv.reshape(-1, 4).astype(np.float32)    tx_iq = trainer.modulator(vv_reshaped, training=False).numpy()        # Kênh truyền    N0 = snr_to_N0(snr_dB)    sigma_ch = np.sqrt(N0 / 2.0)        if getattr(trainer, 'use_pa', False):        tx_complex = tx_iq[:, 0] + 1j * tx_iq[:, 1]        r = np.abs(tx_complex)        theta = np.angle(tx_complex)        a_sat = 2.5        p_rapp = 2.0        ratio = r / a_sat        g_r = r / ((1.0 + ratio**(2*p_rapp))**(1.0/(2*p_rapp)))        pa_tx_complex = g_r * np.exp(1j * theta)        tx_iq = np.stack([pa_tx_complex.real, pa_tx_complex.imag], axis=-1)            # Nhiễu AWGN    noise = np.random.randn(*tx_iq.shape) * sigma_ch    rx_iq = tx_iq + noise        La_bits = np.zeros(num_channel_bits, dtype=np.float32)    bit_errors = []        N0_val = float(N0)    N0_tf = tf.constant(np.full((rx_iq.shape[0], 1), N0_val, dtype=np.float32))    rx_iq_tf = tf.constant(rx_iq, dtype=tf.float32)        for iteration in range(max_iter):        La_interleaved = La_bits[alpha_conv].reshape(-1, 4)        La_int_tf = tf.constant(La_interleaved, dtype=tf.float32)                Le_ext_tf = trainer.demapper([rx_iq_tf, La_int_tf, N0_tf])        Le_ext = Le_ext_tf.numpy().flatten()                Lc = np.zeros(num_channel_bits)        Lc[alpha_conv] = Le_ext                Lc_post = decoder.decode_frame(Lc)        La_bits = scaling_factor * (Lc_post - Lc)                vhat = ((np.sign(Lc_post) + 1) / 2).astype(int)        errors = np.sum(v != vhat)        bit_errors.append(errors)            return bit_errors, num_channel_bitserrs, total_bits = simulate_neural_hamming_bicmid_frame(snr_dB=0.0, max_iter=10)print(f"Bit errors qua 10 iterations (SNR=0.0 dB, total {total_bits} bits):", errs)

## 6. Mô phỏng BER (Neural BICM-ID)

In [ ]:
# ── Quét từ 0 dB đến 7 dB ──dB_range = [0, 1, 2, 3, 4, 5, 6, 7]BER_neural = np.zeros((10, len(dB_range)))log_f.write("\n=== BER Simulation Results (0 to 7 dB) ===\n")print("Đang mô phỏng Neural BICM-ID (0 đến 7 dB)...")for z, snr in enumerate(dB_range):    total_errs = np.zeros(10)    total_bits = 0    blocks = 0        # Chạy nhiều blocks hơn ở SNR cao để đo chính xác BER thấp    if snr <= 2: max_blocks = 200    elif snr <= 4: max_blocks = 500    elif snr <= 5: max_blocks = 1000    else: max_blocks = 2000        while blocks < max_blocks:         errs, n_bits = simulate_neural_hamming_bicmid_frame(snr, max_iter=10)        total_errs += errs        total_bits += n_bits        blocks += 1        for it in range(10):        BER_neural[it, z] = total_errs[it] / total_bits        msg = f"SNR={snr:+.1f}dB | Iter1={BER_neural[0,z]:.2e} Iter5={BER_neural[4,z]:.2e} Iter10={BER_neural[9,z]:.2e} | Blocks={blocks}"    print(msg)    log_f.write(msg + "\n")    log_f.flush()

In [ ]:
plt.figure(figsize=(10, 6))for it_show in [0, 4, 9]:    plt.semilogy(dB_range, np.maximum(BER_neural[it_show, :], 1e-6), '-o', lw=2,                  label=f'Neural BICM-ID (Iter {it_show+1})')plt.grid(True, which='both', alpha=0.3)plt.title('Neural BICM-ID (Extended Hamming 8,4)')plt.xlabel('SNR (dB)')plt.ylabel('BER')plt.ylim(1e-5, 1)plt.legend()plt.show()log_f.write("\nDone.\n")log_f.close()print(f"Log saved to {LOG_FILE}")

## 6. Huấn luyện lại Constellation trên kênh PA (Geometric Shaping)Bật **Power Amplifier (Mô hình Rapp: a_sat=2.5, p=2.0)** và mở khóa Modulator để nó tự do dịch chuyển các điểm chòm sao nhằm "chống lại" sự nén của PA.

In [ ]:
# Bật PA trên channeltrainer.use_pa = True# Mở khóa Modulator để fine-tune Constellationtrainer.modulator.constellation_points._trainable = True# Compile lại với learning rate nhỏ cho toàn mạng (Fine-tuning)trainer.compile(optimizer=tf.keras.optimizers.Adam(1e-4),                loss=tf.keras.losses.BinaryCrossentropy(from_logits=True))print("Tiến hành Fine-tune trên Kênh PA...")FT_EPOCHS = 30ft_losses = []for epoch in range(FT_EPOCHS):    epoch_loss = 0    for _ in range(STEPS_PER_EPOCH):        b_bits, b_snr, b_sigma = generate_batch(2048, 4)        with tf.GradientTape() as tape:            llr_ext = trainer([b_bits, b_snr, b_sigma], training=True)            loss = trainer.compiled_loss(b_bits, llr_ext)                    grads = tape.gradient(loss, trainer.trainable_variables)        trainer.optimizer.apply_gradients(zip(grads, trainer.trainable_variables))        epoch_loss += loss.numpy()            avg_loss = epoch_loss / STEPS_PER_EPOCH    ft_losses.append(avg_loss)    if (epoch+1) % 10 == 0:        print(f"Fine-tune Epoch {epoch+1}/{FT_EPOCHS} - Loss: {avg_loss:.4f}")plt.figure(figsize=(8, 4))plt.plot(ft_losses, color='orange', lw=2)plt.title('Fine-Tuning Loss (PA Channel)')plt.xlabel('Epochs')plt.grid(True)plt.show()

## 7. Trực quan hóa Chòm sao sau Fine-tuningCác điểm chòm sao ở góc (năng lượng cao) có thể bị kéo giãn ra hoặc dịch chuyển để bù trừ cho sự nén của PA.

In [ ]:
# Lấy các điểm đã fine-tunedft_learned_S = trainer.modulator(all_bits, training=False).numpy()plt.figure(figsize=(7, 7))plt.scatter(ft_learned_S[:, 0], ft_learned_S[:, 1], c='green', s=120, zorder=5, label='Fine-tuned (PA)')# Vẽ lại chòm sao ban đầu để so sánhfrom python_code.modulation import build_constellationS_orig = build_constellation(16, [11, 2, 1, 12, 4, 9, 10, 3, 5, 16, 15, 6, 14, 7, 8, 13], 10.0)plt.scatter(S_orig.real, S_orig.imag, c='red', alpha=0.3, s=120, zorder=4, label='Original MSEW')for i in range(16):    bit_str = "".join(str(int(b)) for b in all_bits[i])    plt.annotate(bit_str, (ft_learned_S[i, 0], ft_learned_S[i, 1]), xytext=(5, 5), textcoords='offset points', fontsize=9)plt.title('Fine-tuned Constellation to combat PA (a_sat=2.5)')plt.axhline(0, color='black', lw=1); plt.axvline(0, color='black', lw=1)plt.legend()plt.grid(True, alpha=0.3); plt.axis('equal')plt.show()

## 8. Đánh giá BER của hệ thống Fine-Tuned trên kênh PAChạy lại vòng lặp BICM-ID mô phỏng kênh có PA (a_sat=2.5, p=2.0) với hệ thống mạng đã được Fine-tune.

In [ ]:
# Quét lại BER trên PA channelBER_neural_pa = np.zeros((10, len(dB_range)))print("Đang mô phỏng Neural BICM-ID (trên kênh PA)...")for z, snr in enumerate(dB_range):    total_errs = np.zeros(10)    total_bits = 0    blocks = 0        # Chạy ít blocks hơn xíu để test cho nhanh    if snr <= 2: max_blocks = 200    elif snr <= 4: max_blocks = 500    elif snr <= 5: max_blocks = 1000    else: max_blocks = 1500        while blocks < max_blocks:         errs, n_bits = simulate_neural_hamming_bicmid_frame(snr, max_iter=10)        total_errs += errs        total_bits += n_bits        blocks += 1        for it in range(10):        BER_neural_pa[it, z] = total_errs[it] / total_bits        print(f"PA SNR={snr:+.1f}dB | Iter1={BER_neural_pa[0,z]:.2e} Iter5={BER_neural_pa[4,z]:.2e} Iter10={BER_neural_pa[9,z]:.2e} | Blocks={blocks}")plt.figure(figsize=(10, 6))for it_show in [0, 4, 9]:    plt.semilogy(dB_range, np.maximum(BER_neural_pa[it_show, :], 1e-6), '--s', lw=2,                  label=f'Fine-tuned Neural (Iter {it_show+1}) - PA')plt.grid(True, which='both', alpha=0.3)plt.title('Hiệu năng Fine-tuned Neural BICM-ID trên kênh Nonlinear PA (Rapp, a_sat=2.5)')plt.xlabel('SNR (dB)')plt.ylabel('BER')plt.ylim(1e-5, 1)plt.legend()plt.show()